In [1]:
from pathlib import Path
import pickle
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler

import eda_sol_reweight # with bias to initial distribution
import eda_sol_reweight_noinit # no bias to initial distribution

In [2]:
def load_items(stimuli_dir: Path, trial_id: int) -> np.ndarray:
    csv_path = stimuli_dir / f"civ_items_trial_{trial_id}.csv"
    df = pd.read_csv(csv_path)
    numeric = df.drop(columns=["Name"], errors="ignore")
    return numeric.to_numpy(dtype=np.int64)

def get_eda_params(items: np.ndarray) -> dict:
    n_obj = items.shape[1] - 1

    if n_obj == 3:
        n_selected = 6
        max_row_diff = 5
    elif n_obj == 5:
        n_selected = 10
        max_row_diff = 500 ## should use different criteria for human guided eda (default is 500)
    else:
        raise ValueError(f"Number of objectives {n_obj} not supported")

    return {
        "n_items": items.shape[0],
        "n_obj": n_obj,
        "n_con": 1,
        "n_selected": n_selected,
        "capacity": n_selected * 10,
        "pop_size": 1_000,
        "generations": 100,
        "max_no_improve_gen": 5,
        "max_row_diff": max_row_diff,
    }

def run_eda_pass(items: np.ndarray, params: dict, 
                 seed: int, aspi: np.ndarray, if_rank: bool, temp: float):
    eda_process = eda_sol_reweight_noinit.KnapsackEDA(
        items=items,
        capacity=params["capacity"],
        n_selected=params["n_selected"],
        n_obj=params["n_obj"],
        pop_size=params["pop_size"],
        generations=params["generations"],
        max_no_improve_gen=params["max_no_improve_gen"],
        max_row_diff=params["max_row_diff"],
        seed=seed,
        aspi=aspi,
        if_rank=if_rank,
        temp=temp
    )
    return eda_process.run()

def save_pass_results(run_name: str, results: dict, output_dir: Path) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    file_path = output_dir / f"eda_human_reweight_{run_name}.pkl"

    with open(file_path, "wb") as f:
        pickle.dump(results, f)

    return file_path

In [3]:
def select_test_ref(pf_actual, percentile):
    percentile_unique = np.unique(percentile)
    percent_ref = {}
    for p in percentile_unique:
        qp = np.percentile(pf_actual, p, axis=0)
        percent_ref[p] = qp
    ref_sol = np.array([percent_ref[p][i] for i, p in enumerate(percentile)])
    
    return ref_sol

# def gen_aspi(ref_sol, pf_actual):
#     # normalize by 95 percentile
#     # q95 = np.percentile(pf_actual, 95, axis=0)
#     # aspi = ref_sol/q95

#     # normalize by z-score
#     # aspi = (ref_sol - np.mean(pf_actual, axis=0)) / np.std(pf_actual, axis=0)

#     # min-max quantile
#     q5 = np.percentile(pf_actual, 5, axis=0)
#     q95 = np.percentile(pf_actual, 95, axis=0)
#     aspi = (ref_sol - q5) / (q95 - q5 + 1e-12)

#     # robust scalar (ref_sol - q50) / (q75 - q25)
#     # scaler = RobustScaler()
#     # scaler.fit(pf_actual)
#     # aspi = scaler.transform(ref_sol.reshape(1, -1))[0]

#     return aspi

def gen_aspi(ref_sol, items):
    n_obj = items.shape[1] - 1
    if n_obj == 3:
        n_selected = 6
    elif n_obj == 5:
        n_selected = 10
    else:
        raise ValueError(f"Number of objectives {n_obj} not supported")

    items_q5 = np.percentile(items[:, :n_obj], 5, axis=0)
    items_q95 = np.percentile(items[:, :n_obj], 95, axis=0)
    aspi = (ref_sol - items_q5*n_selected) / (items_q95*n_selected - items_q5*n_selected + 1e-12)
    return aspi 

In [ ]:
# obtain item list
trial_id = 8
stimuli_dir = Path("card_game/stimuli")
items = load_items(stimuli_dir=stimuli_dir, trial_id=trial_id)
with open(f'card_game/eda_results/eda_trial{trial_id}.pkl', 'rb') as f:
    results = pickle.load(f)
pf_actual = results['converged_pf_table'][-1]

# obtain params
params = get_eda_params(items)
n_obj = params["n_obj"]
if_rank = True
temp = 0.3
percentile = np.array([100, 100, 100, 0, 0])

# generate aspiration vector
ref_sol = select_test_ref(pf_actual, percentile)
aspi = gen_aspi(ref_sol, items)
# aspi = np.array([1.0, 1.0, 1.0, 0.0, 0.0])
unit_aspi = aspi / (np.linalg.norm(aspi) + 1e-12)

In [ ]:
# run EDA
run_name = "median_trial8_stan"
results = run_eda_pass(
    items=items,
    params=params,
    seed=1127,
    aspi=unit_aspi,
    if_rank=if_rank,
    temp=temp,
)

# save results
output_dir = Path("data/eda_results/sol_reweight_save_prob/")
save_path = save_pass_results(run_name, results, output_dir)
pf = results["converged_pf_table"][-1]

# save info
file_path = output_dir / f"history_{run_name}.pkl"
history = {
    "trial_id": trial_id,
    "original_aspi": ref_sol,
    "normalized_aspi": aspi,
    "unit_aspi": unit_aspi,
    "if_rank": if_rank,
    "temp": temp
}
with open(file_path, "wb") as f:
    pickle.dump(history, f)

In [ ]:
# run ref x temp pair
temps = np.linspace(0.1, 1, 10)
percentiles = {
    # "qmax": np.array([100, 100, 100, 0, 0]),
    # "qmin": np.array([0, 0, 0, 100, 100]),
    # "qmedian": np.array([50, 50, 50, 50, 50]),
    # "q95": np.array([95, 95, 95, 5, 5]),
    # "q75": np.array([75, 75, 75, 25, 25]),
    # "q60": np.array([60, 60, 60, 40, 40]),
    # "q40": np.array([40, 40, 40, 60, 60]),
    # "q25": np.array([25, 25, 25, 75, 75]),
    # "q5": np.array([5, 5, 5, 95, 95]),
    # "qsame": np.array([75, 75, 75, 75, 75]),
    # "qtest1": np.array([100, 100, 100, 50, 50]),
    # "qtest2": np.array([100, 50, 50, 50, 100]),
    "qtest3": np.array([95, 95, 95, 50, 50]),
    "qtest4": np.array([75, 75, 75, 50, 50]),
}

trial_id = 10
stimuli_dir = Path("card_game/stimuli")
output_dir = Path("data/eda_results/sol_reweight_runs/")
items = load_items(stimuli_dir=stimuli_dir, trial_id=trial_id)
with open(f'card_game/eda_results/eda_trial{trial_id}.pkl', 'rb') as f:
    results = pickle.load(f)
pf_actual = results['converged_pf_table'][-1]

params = get_eda_params(items)
n_obj = params["n_obj"]
if_rank = True

for name, percentile in percentiles.items():
    ref_sol = select_test_ref(pf_actual, percentile)
    aspi = gen_aspi(ref_sol, items)
    unit_aspi = aspi / (np.linalg.norm(aspi) + 1e-12)

    for temp in temps:
        run_name = f"ref{name}_temp{temp:.1f}_trial{trial_id}"
        eda_results = run_eda_pass(
            items=items,
            params=params,
            seed=1127,
            aspi=unit_aspi,
            if_rank=if_rank,
            temp=temp
        )
        save_path = save_pass_results(run_name, eda_results, output_dir)
        file_path = output_dir / f"history_{run_name}.pkl"
        history = {
            "trial_id": trial_id,
            "ref_name": name,
            "percentile": percentile,
            "original_aspi": ref_sol,
            "normalized_aspi": aspi,
            "unit_aspi": unit_aspi,
            "if_rank": if_rank,
            "temp": temp
        }
        with open(file_path, "wb") as f:
            pickle.dump(history, f)

In [ ]:
# load results
run_name = "95_5_trial8_stan_temp03"
trial_id = 8
stimuli_dir = Path("card_game/stimuli")
items = load_items(stimuli_dir=stimuli_dir, trial_id=trial_id)

with open(f"data/eda_results/sol_reweight/eda_human_reweight_{run_name}.pkl", "rb") as f:
    results = pickle.load(f)
pf = results["converged_pf_table"][-1]

with open(f"data/eda_results/sol_reweight/history_{run_name}.pkl", "rb") as f:
    history = pickle.load(f)

with open(f'card_game/eda_results/eda_trial{trial_id}.pkl', 'rb') as f:
    results_actual = pickle.load(f)
pf_actual = results_actual['converged_pf_table'][-1]

In [ ]:
# obtain aspi for plotting
aspi = history["normalized_aspi"]

# normalize pf using min-max quantile
pf_norm = gen_aspi(pf, items)
# pf_norm = gen_aspi(pf, pf_actual)

# compute center solution then normalize it using min-max quantile
center = np.median(pf, axis=0)
dist = np.linalg.norm(pf - center, axis=1) 
center_sol = pf[dist.argmin()]
center_sol_norm = gen_aspi(center_sol, items)
# center_sol_norm = gen_aspi(center_sol, pf_actual)

# normalize pf_actual
pf_actual_norm = gen_aspi(pf_actual, items)
# pf_actual_norm = gen_aspi(pf_actual, pf_actual)

# plot normalized pf, normalized center, and aspiration 
objective_pairs = list(itertools.combinations(range(5), 2)) 
fig, axes = plt.subplots(2, 5, figsize=(24, 8))
axes = axes.ravel()
for ax, (a, b) in zip(axes, objective_pairs):
    ax.plot(pf_actual_norm[:, a], pf_actual_norm[:, b], "go", alpha=0.2, markersize=3, label="PF")
    ax.plot(pf_norm[:, a], pf_norm[:, b], "bo", alpha=0.2, markersize=3, label="PF")
    # sns.kdeplot(
    #     x=pf_actual_norm[:, a],
    #     y=pf_actual_norm[:, b],
    #     levels=10,
    #     fill=True,   
    #     color="green",
    #     ax=ax
    # )
    # sns.kdeplot(
    #     x=pf_norm[:, a],
    #     y=pf_norm[:, b],
    #     levels=10,
    #     fill=True,   
    #     color="steelblue",
    #     alpha=0.4,
    #     ax=ax
    # )
    ax.plot(
        aspi[a], aspi[b],
        "rs", alpha=1, markersize=6, label="Aspiration"
    )
    ax.plot(
        center_sol_norm[a], center_sol_norm[b],
        "ks", alpha=1, markersize=6, label="Center"
    )
    ax.set_xlabel(f"Obj {a + 1}")
    ax.set_ylabel(f"Obj {b + 1}")
    ax.set_xlim(0.1, 0.9)
    ax.set_ylim(0.1, 0.9)
    
fig.tight_layout()
plt.show()

In [ ]:
def plot_ref_temp_panel(
    ax,
    pf,
    pf_actual_norm,
    aspi,
    items,
    obj_a,
    obj_b,
    *,
    xlim=None,
    ylim=None,
):
    """Plot one objective pair for a single ref x temp run."""
    pf_norm = gen_aspi(pf, items)

    center = np.median(pf, axis=0)
    dist = np.linalg.norm(pf - center, axis=1)
    center_sol = pf[dist.argmin()]
    center_sol_norm = gen_aspi(center_sol, items)

    ax.plot(
        pf_actual_norm[:, obj_a], pf_actual_norm[:, obj_b],
        "go", alpha=0.2, markersize=2, label="Actual PF",
    )
    ax.plot(
        pf_norm[:, obj_a], pf_norm[:, obj_b],
        "bo", alpha=0.2, markersize=2, label="EDA PF",
    )
    ax.plot(
        aspi[obj_a], aspi[obj_b],
        "rs", alpha=1, markersize=5, label="Aspiration",
    )
    ax.plot(
        center_sol_norm[obj_a], center_sol_norm[obj_b],
        "ks", alpha=1, markersize=5, label="Center",
    )

    pad = 0.02
    xs = np.concatenate([
        pf_actual_norm[:, obj_a], pf_norm[:, obj_a],
        [aspi[obj_a], center_sol_norm[obj_a]],
    ])
    ys = np.concatenate([
        pf_actual_norm[:, obj_b], pf_norm[:, obj_b],
        [aspi[obj_b], center_sol_norm[obj_b]],
    ])
    ax.set_xlim(*(xlim if xlim is not None else (xs.min() - pad, xs.max() + pad)))
    ax.set_ylim(*(ylim if ylim is not None else (ys.min() - pad, ys.max() + pad)))


def plot_ref_temp_grid(
    obj_a: int,
    obj_b: int,
    *,
    trial_id: int = 8,
    stimuli_dir: Path = Path("card_game/stimuli"),
    output_dir: Path = Path("data/eda_results/sol_reweight_runs/"),
    percentiles: dict | None = None,
    temps: np.ndarray | None = None,
    xlim=None,
    ylim=None,
    save_path: Path | None = None,
):
    """Grid of ref x temp panels for one objective pair."""
    if percentiles is None:
        percentiles = {
            "qmax": np.array([100, 100, 100, 0, 0]),
            "qmin": np.array([0, 0, 0, 100, 100]),
            "qmedian": np.array([50, 50, 50, 50, 50]),
            "q95": np.array([95, 95, 95, 5, 5]),
            "q75": np.array([75, 75, 75, 25, 25]),
            "q60": np.array([60, 60, 60, 40, 40]),
            "q40": np.array([40, 40, 40, 60, 60]),
            "q25": np.array([25, 25, 25, 75, 75]),
            "q5": np.array([5, 5, 5, 95, 95]),
            "qsame": np.array([75, 75, 75, 75, 75]),
            # "qoppo": np.array([95, 5, 5, 95, 5]),
            # "qoppo2": np.array([95, 95, 5, 95, 5]),
            "qtest1": np.array([100, 100, 100, 50, 50]),
            "qtest2": np.array([100, 50, 50, 50, 100])
        }
    if temps is None:
        temps = np.linspace(0.1, 1, 10)

    items = load_items(stimuli_dir=stimuli_dir, trial_id=trial_id)
    with open(f"card_game/eda_results/eda_trial{trial_id}.pkl", "rb") as f:
        pf_actual = pickle.load(f)["converged_pf_table"][-1]
    pf_actual_norm = gen_aspi(pf_actual, items)

    ref_names = list(percentiles.keys())
    n_rows, n_cols = len(ref_names), len(temps)
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(n_cols * 2.4, n_rows * 2.4),
        squeeze=False,
    )

    for i, name in enumerate(ref_names):
        for j, temp in enumerate(temps):
            run_name = f"ref{name}_temp{temp:.1f}_trial{trial_id}"
            with open(output_dir / f"eda_human_reweight_{run_name}.pkl", "rb") as f:
                results = pickle.load(f)
            with open(output_dir / f"history_{run_name}.pkl", "rb") as f:
                history = pickle.load(f)

            pf = results["converged_pf_table"][-1]
            aspi = history["normalized_aspi"]
            ax = axes[i, j]

            plot_ref_temp_panel(
                ax, pf, pf_actual_norm, aspi, items, obj_a, obj_b,
                xlim=xlim, ylim=ylim,
            )

            if i == 0:
                ax.set_title(f"temp={temp:.1f}", fontsize=9)
            if j == 0:
                ax.set_ylabel(f"{name}\nObj {obj_b}", fontsize=9)
            if i == n_rows - 1:
                ax.set_xlabel(f"Obj {obj_a}", fontsize=9)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.02))
    fig.suptitle(
        f"Obj {obj_a} vs Obj {obj_b} (trial {trial_id})",
        y=1.05,
        fontsize=12,
    )
    fig.tight_layout()

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches="tight")

    plt.show()
    return fig


# specify objective pair to plot (0-indexed)
obj_a, obj_b = 0, 1
trial_id = 10
plot_ref_temp_grid(
    obj_a,
    obj_b,
    trial_id=trial_id,
    save_path=Path(f"data/eda_results/sol_reweight_runs/ref_temp_grid_obj{obj_a}_obj{obj_b}_trial{trial_id}.png"),
)

In [ ]:
sns.heatmap(np.corrcoef(items.T), annot=True, cmap="coolwarm")
plt.show()

In [ ]:
import math

if "distribution_table" not in results:
    raise KeyError(
        "results has no 'distribution_table'. Re-run the EDA cell after updating "
        "eda_sol_reweight_noinit.py, or load a newer results pickle."
    )

distribution_table = np.vstack(results["distribution_table"])
mode1_gens = results["mode 1 generations"]
n_gens, n_items_dist = distribution_table.shape

item_names = pd.read_csv(stimuli_dir / f"civ_items_trial_{trial_id}.csv")["Name"].tolist()
x_items = np.arange(n_items_dist)


def plot_distribution_evolution(step: int = 1):
    """Overlay item-probability distributions across generations."""
    gen_indices = list(range(0, n_gens, step))
    colors = plt.cm.viridis_r(np.linspace(0, 1, len(gen_indices)))

    fig, ax = plt.subplots(figsize=(12, 5))
    for color, gen_idx in zip(colors, gen_indices):
        phase = "mode 1" if gen_idx < mode1_gens else "mode 2"
        ax.plot(
            x_items,
            distribution_table[gen_idx],
            color=color,
            alpha=0.7,
            linewidth=0.9,
            label=f"gen {gen_idx + 1} ({phase})" if gen_idx in (gen_indices[0], gen_indices[-1]) else None,
        )

    ax.set_xlabel("Item index")
    ax.set_ylabel("Probability")
    ax.set_title(f"Distribution evolution ({len(gen_indices)} generations)")
    sm = plt.cm.ScalarMappable(cmap=plt.cm.viridis_r, norm=plt.Normalize(1, n_gens))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, pad=0.01)
    cbar.set_label("Generation")
    ax.legend(loc="upper right")
    fig.tight_layout()
    plt.show()

def plot_distribution_grid(step: int = 5, top_k: int = 10):
    """Bar charts of top-k items at selected generations."""
    gen_indices = list(range(0, n_gens, step))
    n_plots = len(gen_indices)
    cols = 5
    rows = math.ceil(n_plots / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3.5))
    axes = np.atleast_1d(axes).ravel()

    for k, gen_idx in enumerate(gen_indices):
        ax = axes[k]
        probs = distribution_table[gen_idx]
        top_idx = np.argsort(probs)[-top_k:][::-1]
        ax.bar(range(top_k), probs[top_idx], color="steelblue", alpha=0.85)
        ax.set_xticks(range(top_k))
        ax.set_xticklabels([item_names[i] for i in top_idx], rotation=60, ha="right", fontsize=7)
        phase = "mode 1" if gen_idx < mode1_gens else "mode 2"
        ax.set_title(f"gen {gen_idx + 1} ({phase})")
        ax.set_ylabel("Probability")

    for ax in axes[n_plots:]:
        ax.axis("off")

    fig.suptitle(f"Top {top_k} items by probability", y=1.02)
    fig.tight_layout()
    plt.show()


def plot_js_divergence():
    """JS divergence across generations."""
    js_div_list = results["js_div_list"]
    gens = np.arange(1, len(js_div_list) + 1)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(gens, js_div_list, "o-", markersize=3, linewidth=1)
    ax.axvline(x=mode1_gens, color="red", linestyle="--", linewidth=1, label="mode 1 | mode 2")
    ax.set_xlabel("Generation")
    ax.set_ylabel("JS divergence")
    ax.set_title("Distribution convergence (JS divergence)")
    ax.legend()
    fig.tight_layout()
    plt.show()


plot_distribution_evolution(step=1)
# plot_distribution_grid(step=max(1, n_gens // 10))
plot_js_divergence()

In [ ]:
import math

n_obj = items.shape[1] - 1
objective_pairs = list(itertools.combinations(range(n_obj), 2))
mode1_gens = results["mode 1 generations"]
n_gens = len(results["pareto_front_table"])

# normalize reference pf_actual for background
pf_actual_norm = gen_aspi(pf_actual, items)
aspi_plot = history["normalized_aspi"]

def normalize_front(front: np.ndarray) -> np.ndarray:
    """Normalize objective values; handle pareto_front's extra constraint column."""
    obj = front[:, :n_obj] if front.shape[1] > n_obj else front
    return gen_aspi(obj, items)

def plot_generation_grid(
    obj_a: int,
    obj_b: int,
    *,
    step: int = 1,
    show_actual: bool = True,
):
    """Plot pareto_front_table and converged_pf_table for each generation."""
    gen_indices = list(range(0, n_gens, step))
    n_plots = len(gen_indices)
    cols = 5
    rows = math.ceil(n_plots / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3.5))
    axes = np.atleast_1d(axes).ravel()

    for k, gen_idx in enumerate(gen_indices):
        ax = axes[k]

        if show_actual:
            ax.plot(
                pf_actual_norm[:, obj_a],
                pf_actual_norm[:, obj_b],
                "go",
                alpha=0.15,
                markersize=2,
                label="Actual PF" if k == 0 else None,
            )

        pf = normalize_front(results["pareto_front_table"][gen_idx])
        ax.plot(
            pf[:, obj_a],
            pf[:, obj_b],
            marker="o",
            linestyle="None",
            markerfacecolor="none",
            markeredgecolor="k",
            markersize=3,
            alpha=0.7,
            label="Pareto front" if k == 0 else None,
        )

        conv_idx = gen_idx - mode1_gens
        if conv_idx >= 0:
            cpf = normalize_front(results["converged_pf_table"][conv_idx])
            ax.plot(
                cpf[:, obj_a],
                cpf[:, obj_b],
                "rs",
                alpha=0.25,
                markersize=2,
                label="Converged PF" if k == 0 else None,
            )

        ax.plot(
            aspi_plot[obj_a],
            aspi_plot[obj_b],
            "r*",
            markersize=8,
            label="Aspiration" if k == 0 else None,
        )

        phase = "mode 1" if gen_idx < mode1_gens else "mode 2"
        ax.set_title(f"gen {gen_idx + 1} ({phase})")
        ax.set_xlabel(f"Obj {obj_a + 1}")
        ax.set_ylabel(f"Obj {obj_b + 1}")

    for ax in axes[n_plots:]:
        ax.axis("off")

    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.02))
    fig.suptitle(f"Obj {obj_a + 1} vs Obj {obj_b + 1}", y=1.05)
    fig.tight_layout()
    plt.show()

# overview: obj 1 vs obj 2 for every generation
plot_generation_grid(0, 1, step=1)

In [ ]:
def calculate_dominated(pf_predicted, pf_actual, n_obj):
    dominated = np.zeros(len(pf_predicted))
    for i in range(len(pf_predicted)):
        for j in range(len(pf_actual)):
            if np.all(pf_actual[j, :n_obj] >= pf_predicted[i, :n_obj]) and \
                np.any(pf_actual[j, :n_obj] > pf_predicted[i, :n_obj]):
                dominated[i] = 1
                break
    return dominated, np.sum(dominated), np.sum(dominated)/len(pf_predicted)
dominated, num_dominated, ratio_dominated = calculate_dominated(pf_actual, pf, 5)
print(ratio_dominated)